In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
tdf_riders = pd.read_csv('tdf_riders.csv')

In [3]:
X = tdf_riders.iloc[:, 2:5]
y = tdf_riders['General Classification']

In [ ]:
# Data Analysis

# X_country_freq = pd.DataFrame({
#     'Frequency': value_counts.values,
#     'Original Label': label_encoder.inverse_transform(value_counts.index)
# })

# X_country_freq.sort_values(by='Frequency', ascending=False)

In [4]:
# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

In [6]:
class CyclistNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(3, 8)
        self.relu = nn.ReLU()  # Add activation function
        self.layer2 = nn.Linear(8, 1)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)  # Apply activation
        x = self.layer2(x)
        return x


In [7]:
model = CyclistNN()
criterion = nn.MSELoss()  # Mean Squared Error Loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer with learning rate 0.001

In [8]:
num_epochs = 1000
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    # Backward pass and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [100/1000], Loss: 7896680.0000
Epoch [200/1000], Loss: 7892753.5000
Epoch [300/1000], Loss: 7886275.0000
Epoch [400/1000], Loss: 7877006.5000
Epoch [500/1000], Loss: 7865182.0000
Epoch [600/1000], Loss: 7850563.0000
Epoch [700/1000], Loss: 7832117.0000
Epoch [800/1000], Loss: 7809644.0000
Epoch [900/1000], Loss: 7783417.0000
Epoch [1000/1000], Loss: 7753233.0000


In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor)
    test_loss = criterion(y_pred, y_test_tensor)
    print(f'Test Loss: {test_loss.item():.4f}')

Test Loss: 7230874.0000


In [20]:
weights = model.layer1.weight.data.numpy()
feature_importance = np.abs(weights).mean(axis=0)
for feature, importance in zip(X.columns, feature_importance):
    print(f'{feature}: {importance:.4f}')

TT: 1.4237
Sprint: 1.1684
Climb: 1.4353


In [21]:
# Get feature names
feature_names = X.columns

# Calculate relative importance
total_importance = feature_importance.sum()
relative_importance = feature_importance / total_importance

# This will show you the "ratio" you're looking for
for name, importance, relative in zip(feature_names, feature_importance, relative_importance):
    print(f"{name}:")
    print(f"  Absolute importance: {importance:.4f}")
    print(f"  Relative importance: {relative:.2%}")


TT:
  Absolute importance: 1.4237
  Relative importance: 35.35%
Sprint:
  Absolute importance: 1.1684
  Relative importance: 29.01%
Climb:
  Absolute importance: 1.4353
  Relative importance: 35.64%
